In [ ]:
%matplotlib inline

# Calibrate an ML model

## Problem

My ML model is not good enough.
How can I calibrate its hyperparameters to improve its quality?

## Solution

Use the
[MLModelCalibration][gemseo.machine_learning.core.calibration.MLModelCalibration] algorithm.

## Step-by-step guide


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt

from gemseo.algos.design_space import DesignSpace
from gemseo.algos.doe.pydoe.settings.pydoe_fullfact import PYDOE_FULLFACT_Settings
from gemseo.machine_learning.core.calibration import MLModelCalibration
from gemseo.machine_learning.regression.models.polyreg_settings import (
    PolynomialRegressor_Settings,
)
from gemseo.machine_learning.regression.quality.mse_measure import MSEMeasure
from gemseo.problems.dataset.rosenbrock import create_rosenbrock_dataset

### 1. Create the training dataset



In [ ]:
training_dataset = create_rosenbrock_dataset(opt_naming=False, n_samples=25)

### 2. Create the test training




In [ ]:
test_dataset = create_rosenbrock_dataset(opt_naming=False)

### 3. Calibrate the degree of a polynomial regressor

#### Create the calibration space



In [ ]:
calibration_space = DesignSpace()
calibration_space.add_variable("degree", 1, "integer", 1, 10, 1)

#### Instantiate the calibration algorithm



In [ ]:
calibration = MLModelCalibration(
    PolynomialRegressor_Settings(),
    training_dataset,
    calibration_space,
    MSEMeasure,
    measure_evaluation_method_name="TEST",
    measure_options={"test_data": test_dataset},
)

#### Execute the calibration algorithm



In [ ]:
calibration.execute(PYDOE_FULLFACT_Settings(n_samples=10))

#### Get the main calibration results



In [ ]:
x_opt = calibration.optimal_parameters
f_opt = calibration.optimal_criterion
degree = x_opt["degree"][0]
f"optimal degree = {degree}; optimal criterion = {f_opt}"

#### Get the calibration history



In [ ]:
calibration.dataset

#### Visualize the calibration history



In [ ]:
degree = calibration.get_history("degree")
criterion = calibration.get_history("criterion")
learning = calibration.get_history("learning")

plt.plot(degree, criterion, "-o", label="test", color="red")
plt.plot(degree, learning, "-o", label="learning", color="blue")
plt.xlabel("polynomial degree")
plt.ylabel("quality")
plt.axvline(x_opt["degree"], color="red", ls="--")
plt.legend()

## Summary

An ML model can be calibrated using the
[MLModelCalibration][gemseo.machine_learning.core.calibration.MLModelCalibration] algorithm,
fed by an ML model name,
a training dataset,
a quality measure and a driver.

